Direct YML Files analysis or keywords detection


In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import pandas as pd
from typing import List, Pattern, Tuple

# === CONFIGURATION ===
CONFIG_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_YML_Files.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
Search_Method_Name = "Phase 1 - Direct YAML Scan (tightened sdkmanager + weak-hint gating + emulator-vs-real reconcile + stricter instru flag)"

# --- helpers ---
def compile_any(patterns: List[str], flags=re.I | re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: List[Pattern], text: str) -> bool:
    return any(p.search(text) for p in patterns)

def unique_preserve(seq: List[str]) -> List[str]:
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

# --- strip comments (analyze only active lines) ---
COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')
def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

# === Define keyword SOURCES with explicit GROUPS ===
# NOTE: 'actions/setup-android' is intentionally NOT treated as device setup by itself.
DEVICE_SOURCES: List[Tuple[str, str, List[str]]] = [
    # Real devices
    ("Real_Device", "adb devices",      [r'(?m)^\s*adb\s+devices\b']),
    ("Real_Device", "adb get-state",    [r'(?m)^\s*adb\s+get-state\b']),
    ("Real_Device", "adb get-serialno", [r'(?m)^\s*adb\s+get-serialno\b']),
    ("Real_Device", "adb -s <serial> (physical)", [r'(?m)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b']),
    ("Real_Device", "adb install",      [r'(?m)^\s*adb\s+install(\s+-r)?\b']),
    ("Real_Device", "adb shell",        [r'(?m)^\s*adb\s+shell\b']),
    ("Real_Device", "adb root",         [r'(?m)^\s*adb\s+root\b']),
    ("Real_Device", "adb settings",     [r'(?m)^\s*adb\s+shell\s+settings\b']),
    ("Real_Device", "adb input",        [r'(?m)^\s*adb\s+shell\s+input\b']),
    ("Real_Device", "adb pm grant",     [r'(?m)^\s*adb\s+shell\s+pm\s+grant\b']),

    ("Emulator", "adb -s emulator-serial", [
    r'(?m)^\s*adb\s+-s\s+emulator-\d+\b',
    r'(?m)^\s*adb\s+-s\s+(?:localhost|127\.0\.0\.1):\d+\b',
    ]),


    

    # Emulators / managed virtual devices
    # STRONG emulator signals (real setup/launch):
    ("Emulator", "emulator -avd/@", [
    r'(?m)^\s*\S*emulator\b[^\n]*\s(-avd|@)\S+'
    ]),
    ("Emulator", "android-wait-for-emulator", [
    r'(?m)^\s*(?:\./)?android-wait-for-emulator\b'
    ]),
    ("Emulator", "start-emulator.sh",         [r'(?m)^\s*start-emulator\.sh\b']),
    ("Emulator", "android create avd", [r'(?m)^\s*\S*android\b[^\n]*\bcreate\s+avd\b']),
    ("Emulator", "circle-android wait-for-boot",[r'(?m)^\s*circle-android\s+wait-for-boot\b']),
    ("Emulator", "reactivecircus runner",     [r'uses:\s*reactivecircus/android-emulator-runner']),
    # WEAK emulator signals (gated below):
    ("Emulator", "sys-img component", [
    r'(?m)^\s*-\s*sys-img-[^\s]*-android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
    r'(?m)^\s*-\s*sys-img-[^\s]*-google_apis-[^\s]*(?:\d+|\$[A-Z_][A-Z0-9_]*)\b'
    ]),

    # avdmanager itself is a strong signal:
    ("Emulator", "avdmanager",     [r'(?m)^\s*\S*avdmanager\b']),
    # sdkmanager counts ONLY when installing system-images or emulator package:
    ("Emulator", "sdkmanager system-images/emulator", [
    r'(?m)^\s*\S*sdkmanager\b[^\n"]*"system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)[^"\n]*"|'
    r'(?m)^\s*\S*sdkmanager\b[^\n]*\bsystem-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b'
    ]),
    # Typical AVD specs (WEAK hints; gated below):
    ("Emulator", "api-level",                 [r'\bapi[-_ ]?level\b\s*:?\s*\d{2}']),
    ("Emulator", "abi/arch",                  [r'\b(abi|arch)\b\s*:?\s*(x86|x86_64|arm64|armeabi)']),
    ("Emulator", "target image",              [r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)']),
    ("Emulator", "device name",               [r'\b(avd[-_ ]?name|device)\b\s*:\s*pixel']),

    # Gradle Managed Devices (GMD)
    ("GMD", "managedDevices DSL",             [r'\bmanageddevices?\b']),
    ("GMD", "ManagedVirtualDevice DSL",       [r'\bmanagedvirtualdevice\b|\bcom\.android\.build\.api\.dsl\.ManagedVirtualDevice\b']),
    ("GMD", "GMD task mentions",              [r'\bmanageddevice\w*androidtest\b']),
    ("GMD", "GHA gradle arguments/tasks",     [
        r'(?m)^\s*arguments\s*:\s*[:\w-]*manageddevice\w*androidtest\b',
        r'(?m)^\s*tasks?\s*:\s*[:\w-]*manageddevice\w*androidtest\b'
    ]),

    # Third-party device labs
    ("Third_Party_Lab", "gcloud firebase",    [r'(?m)^\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",           [r'(?m)^\s*saucectl(\s+run)?\b']),
    ("Third_Party_Lab", "browserstack/bstack",[r'\b(browserstack|bstack)\b']),
    ("Third_Party_Lab", "appcenter test",     [r'(?m)^\s*appcenter\s+test\s+run\s+android\b']),
    ("Third_Party_Lab", "maestro cloud",      [r'(?m)^\s*maestro\s+cloud\b']),
    ("Third_Party_Lab", "test_matrix/firebase.json", [r'\btest_matrix\.json\b|\bfirebase\.json\b']),
]

TRIGGER_SOURCES: List[Tuple[str, str, List[str]]] = [
    ("Gradle", "connectedAndroidTest",   [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+(:[\w-]+:)?connectedandroidtest\b']),
    ("Gradle", "connectedAndroidTest (abbr)", [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?[^\n]*\b(?:cat|connectedandroidtest)\b']),
    ("Gradle", "connected.*Android.*",   [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+(:[\w-]+:)?connected.*android.*test\b']),
    ("Gradle", "connectedCheck",         [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+(:[\w-]+:)?connectedcheck\b']),
    ("Gradle", "createInstrCoverage",    [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+.*createinstrumentationtestcoveragereport\b']),
    ("Gradle", "runInstrumentationTests",[r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+.*runinstrumentationtests\b']),
    ("Gradle", "executeScreenshotTests", [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+.*executescreenshottests\b']),
    ("Gradle", "orchestrator task",      [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+.*orchestrator\b']),
    ("Gradle", "yaml script -> gradle",  [r'\bscript\s*:\s*(?:\./|\.\\)?gradlew(?:\.bat)?\s+(:[\w-]+:)?connected.*']),
    ("Gradle", "connected (broad)",      [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+(:[\w-]+:)?connected.*\b']),
    ("ADB",    "am instrument",          [r'(?m)^\s*(?:sudo\s+)?(?:adb\s+shell\s+)?am\s+instrument\b']),
    ("Third_Party_Lab", "gcloud firebase",[r'(?m)^\s*(?:sudo\s+)?gcloud(?:\s+beta)?(?:\s+--quiet)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",      [r'(?m)^\s*(?:sudo\s+)?saucectl(?:\s+run)?\b']),
    ("Third_Party_Lab", "appcenter run", [r'(?m)^\s*(?:sudo\s+)?appcenter\s+test\s+run\s+android\b']),
    ("Flutter", "flutter drive", [r'(?m)^\s*(?:sudo\s+)?flutter\s+drive\b']),
    ("Flutter", "flutter test (integration_test)", [r'(?m)^\s*(?:sudo\s+)?flutter\s+test\b[^\n]*\bintegration_test\b']),
    # (Optional) dart test for integration flows in some repos
    ("Flutter", "dart test (integration_test)", [r'(?m)^\s*(?:sudo\s+)?dart\s+test\b[^\n]*\bintegration_test\b']),

]

# GH Actions gradle/gradle-build-action inputs
GHA_GRADLE_INPUTS = compile_any([
    r'(?m)^\s*arguments\s*:\s*[:\w-]*connected.*android.*test\b',
    r'(?m)^\s*arguments\s*:\s*[:\w-]*manageddevice\w*androidtest\b',
    r'(?m)^\s*arguments\s*:\s*connectedcheck\b',
    r'(?m)^\s*tasks?\s*:\s*[:\w-]*connected.*android.*test\b',
])

# Precompile

DEVICE_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]
TRIGGER_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES]

def collect_hits_with_groups(patterns: List[Tuple[str, str, List[Pattern]]], text: str):
    labels, groups = [], []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl); groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

# --- WEAK-HINT GATING ---
STRONG_DEVICE_LABELS = {
    # Emulator (strong)
    "emulator -avd/@", "android-wait-for-emulator", "start-emulator.sh",
    "circle-android wait-for-boot", "reactivecircus runner",
    "avdmanager", "sdkmanager system-images/emulator","android create avd",
    # GMD
    "managedDevices DSL", "ManagedVirtualDevice DSL", "GMD task mentions", "GHA gradle arguments/tasks",
    # Real device / labs
    "adb get-state", "adb get-serialno", "adb -s <serial>",
    "gcloud firebase", "saucectl", "browserstack/bstack", "appcenter test", "maestro cloud",
    "test_matrix/firebase.json",
}
WEAK_DEVICE_LABELS = {"api-level", "abi/arch", "target image", "device name", "sys-img component"}

def filter_weak_device_hints(labels, groups):
    """Remove weak hints unless at least one strong device signal is present."""
    if not (set(labels) & STRONG_DEVICE_LABELS):
        labels = [l for l in labels if l not in WEAK_DEVICE_LABELS]
        if not labels:
            groups = []
    return labels, groups

# --- EMULATOR vs REAL DEVICE RECONCILIATION ---
EMULATOR_STRONG_LABELS = {
    "emulator -avd/@", "android-wait-for-emulator", "start-emulator.sh",
    "circle-android wait-for-boot", "reactivecircus runner", "avdmanager",
    "sdkmanager system-images/emulator","adb -s emulator-serial"
}
REAL_DEVICE_STRONG_LABELS = {
     "adb get-state", "adb get-serialno", "adb -s <serial> (physical)"
}
REAL_DEVICE_GENERIC_ADB = {
    "adb devices","adb install", "adb shell", "adb root", "adb settings", "adb input", "adb pm grant"
}

def reconcile_emulator_vs_real(labels, groups):
    """
    If strong emulator signals exist and Real_Device is only due to generic ADB,
    drop those generic ADB labels and potentially the Real_Device group.
    """
    lbls = set(labels)
    has_emulator_strong = bool(lbls & EMULATOR_STRONG_LABELS)
    if has_emulator_strong:
        # remove generic ADB labels
        lbls -= REAL_DEVICE_GENERIC_ADB
        labels = [l for l in labels if l in lbls]
        # if Real_Device remains only due to strong real-device hints keep it; else drop group
        if "Real_Device" in groups:
            has_real_after = bool(set(labels) & REAL_DEVICE_STRONG_LABELS)
            if not has_real_after:
                groups = [g for g in groups if g != "Real_Device"]
    return labels, groups


def hard_emulator_priority(device_labels, device_groups):
    # if emulator is present and Real_Device is present
    # but no strong Real_Device labels are present, drop Real_Device
    if ("Emulator" in device_groups
        and "Real_Device" in device_groups
        and not (set(device_labels) & REAL_DEVICE_STRONG_LABELS)):
        device_groups = [g for g in device_groups if g != "Real_Device"]
        device_labels = [l for l in device_labels if l not in REAL_DEVICE_GENERIC_ADB]
    return device_labels, device_groups

# --- post-filters scaffold (kept for future use) ---
def drop_lonely_setup_android(device_labels, device_groups):
    # No-op placeholder; kept for future weak-hint toggles if needed.
    return device_labels, device_groups

# === scan & export ===
rows = []

for fname in os.listdir(CONFIG_DIR):
    ext = os.path.splitext(fname)[1].lower()
    if ext not in ('.yml', '.yaml'):
        continue

    fpath = os.path.join(CONFIG_DIR, fname)
    if not os.path.isfile(fpath):
        continue

    with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()

    # ---------- Primary pass ----------
    content = strip_comments(raw).lower()
    content = re.sub(r'(?m)^\s*-\s*', '', content)  # normalize list bullets so ^-anchored patterns hit

    device_labels, device_groups = collect_hits_with_groups(DEVICE_PATTERNS, content)
    trigger_labels, trigger_groups = collect_hits_with_groups(TRIGGER_PATTERNS, content)

    # GATE WEAK HINTS + RECONCILE EMULATOR vs REAL
    device_labels, device_groups = filter_weak_device_hints(device_labels, device_groups)
    device_labels, device_groups = reconcile_emulator_vs_real(device_labels, device_groups)

    device_labels, device_groups = hard_emulator_priority(device_labels, device_groups)

    # post-filter scaffold
    device_labels, device_groups = drop_lonely_setup_android(device_labels, device_groups)

    # ---------- Fallback pass (only if primary found nothing) ----------
    fallback_detected = False
    if not device_labels and not trigger_labels:
        fallback = strip_comments(raw).lower()
        fallback = re.sub(r'(?m)^\s*-\s*', '', fallback)                          # bullets
        fallback = re.sub(r'(?m)^\s*(?:command|run|script)\s*:\s*', '', fallback) # hide keys
        fallback = re.sub(r'(?m)^\s*sudo\s+', '', fallback)                        # leading sudo

        fb_device_labels, fb_device_groups = collect_hits_with_groups(DEVICE_PATTERNS, fallback)
        fb_trigger_labels, fb_trigger_groups = collect_hits_with_groups(TRIGGER_PATTERNS, fallback)

        # GATE WEAK HINTS + RECONCILE in fallback too
        fb_device_labels, fb_device_groups = filter_weak_device_hints(fb_device_labels, fb_device_groups)
        fb_device_labels, fb_device_groups = reconcile_emulator_vs_real(fb_device_labels, fb_device_groups)

        # gradle-build-action inputs -> treat as trigger
        if any_match(GHA_GRADLE_INPUTS, fallback):
            fb_trigger_labels.append("gha gradle arguments")
            fb_trigger_groups.append("Gradle")

        if fb_device_labels or fb_trigger_labels:
            fallback_detected = True
            device_labels  = unique_preserve(device_labels  + fb_device_labels)
            device_groups  = unique_preserve(device_groups  + fb_device_groups)
            trigger_labels = unique_preserve(trigger_labels + fb_trigger_labels)
            trigger_groups = unique_preserve(trigger_groups + fb_trigger_groups)

            device_labels, device_groups = hard_emulator_priority(device_labels, device_groups)

    # parse full_name and ci_platform from: <full_name>__<platform>++<file>.yml
    full_name = "Unknown"
    ci_platform = "Unknown"
    base = os.path.basename(fname)
    if "__" in base and "++" in base:
        try:
            full_name = base.split("__", 1)[0]
            ci_platform = base.split("__", 1)[1].split("++", 1)[0]
        except Exception:
            pass

    has_device_setup = bool(device_labels)
    has_test_trigger = bool(trigger_labels)

    # === stricter instrumentation flag ===
    # instru_t_ci = trigger present OR (device setup present AND it's a real execution venue)
    real_device_groups = {"Emulator", "GMD", "Third_Party_Lab", "Real_Device"}
    has_real_device_group = any(g in real_device_groups for g in device_groups)
    instru_t_ci = bool(has_test_trigger or (has_device_setup and has_real_device_group))

    rows.append({
        "filename": fname,
        "full_name": full_name,
        "ci_platform": ci_platform,
        "has_device_setup": has_device_setup,
        "device_setup": ", ".join(device_labels),
        "device_setup_group": ", ".join(device_groups),
        "has_test_trigger": has_test_trigger,
        "test_trigger": ", ".join(trigger_labels),
        "test_trigger_group": ", ".join(trigger_groups),
        "instru_t_ci": instru_t_ci,
        "Search_Method_Name": Search_Method_Name,
        "fall_back": bool(fallback_detected),
    })

# Export
df = pd.DataFrame(rows, columns=[
    "filename", "full_name", "ci_platform",
    "has_device_setup", "device_setup", "device_setup_group",
    "has_test_trigger", "test_trigger", "test_trigger_group",
    "instru_t_ci",
    "Search_Method_Name",
    "fall_back",
])
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV} (yaml files={len(df)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_YML_Files.csv (yaml files=12667)
